# ♟️ Chess Numba — 300k Nodes 實戰對戰驗收 (Tournament Runner)

本 Notebook 用於在 Google Colab 雲端高並行執行完整對抗錦標賽：
```bash
python tools/tournament.py --games 400 --nodes 300000 --concurrency 12 --tt-mb 64
```

### 🎯 評估目標
* **Engine 1 (New)**: `main.py`（最新 SPSA 60 輪優化參數 + SF19 深度自適應除數保護 + Plan A 歷史懲罰安全水位）
* **Engine 2 (Old)**: `main_old.py`（`classical_old` 基準引擎）
* **時控規範**: 300,000 nodes（長時控深度視野，平均 depth 14~18）
* **對戰規格**: 400 局（200 對開局交換白黑），64MB 轉置表，開局庫 `data/openings.epd`


In [6]:
# 1. 檢視 Colab 雲端硬體與 CPU 核心數
!lscpu | grep "Model name\|Socket\|Thread\|NUMA\|CPU(s)"
!free -h
!python3 --version


CPU(s):                                  8
On-line CPU(s) list:                     0-7
Model name:                              AMD EPYC 7B12
Thread(s) per core:                      2
Socket(s):                               1
NUMA node(s):                            1
NUMA node0 CPU(s):                       0-7
               total        used        free      shared  buff/cache   available
Mem:            50Gi       1.4Gi        45Gi       2.2Mi       4.4Gi        49Gi
Swap:             0B          0B          0B
Python 3.13.16


In [11]:
# 2. 克隆/拉取最新代碼庫並安裝相依套件
import os
branch = 'feat-initial-bitboard-setup'

if not os.path.exists('/content/chess_numba'):
    %cd /content
    !git clone -b {branch} https://github.com/b08203031/chess_numba.git

%cd /content/chess_numba
!git fetch origin {branch}
!git checkout -f {branch}
!git reset --hard origin/{branch}
!git clean -fd
!pip install -q numba chess numpy


/content/chess_numba
remote: Enumerating objects: 5, done.
remote: Counting objects: 100% (5/5), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 3 (delta 2), reused 3 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (3/3), 969 bytes | 969.00 KiB/s, done.
From https://github.com/b08203031/chess_numba
 * branch            feat-initial-bitboard-setup -> FETCH_HEAD
   6ae5d18..99f196f  feat-initial-bitboard-setup -> origin/feat-initial-bitboard-setup
Already on 'feat-initial-bitboard-setup'
Your branch is behind 'origin/feat-initial-bitboard-setup' by 1 commit, and can be fast-forwarded.
  (use "git pull" to update your local branch)
HEAD is now at 99f196f fix(colab): invalidate Python module cache and force-clean constants in tournament runner


In [12]:
# 3. 檢驗最新常數與 SF19 深度保護（Plan A: SPSA60 + SF19保護 + Malus=1600）
import sys
from pathlib import Path

# 1) 清理 Python 模組記憶體快取，確保重新從磁碟載入最新代碼
for m in list(sys.modules.keys()):
    if 'chess_engine' in m or 'tune_search' in m:
        del sys.modules[m]

# 2) 雙重保險：強制確保磁碟上的 TUNE_LMR_BASE_OFFSET 為 6
const_file = Path('chess_engine/classical/constants.py')
if const_file.exists():
    text = const_file.read_text(encoding='utf-8')
    if 'TUNE_LMR_BASE_OFFSET = 449' in text:
        text = text.replace('TUNE_LMR_BASE_OFFSET = 449', 'TUNE_LMR_BASE_OFFSET = 6')
        const_file.write_text(text, encoding='utf-8')

import chess_engine.classical.constants as C

print('✅ 引擎目前生效參數檢驗：')
print(f'  TUNE_LMR_BASE_OFFSET (slot index): {C.TUNE_LMR_BASE_OFFSET}')
print(f'  LMR_BASE_OFFSET: {C.LMR_BASE_OFFSET}')
print(f'  LMR_MOVECOUNT_FACTOR: {C.LMR_MOVECOUNT_FACTOR}')
print(f'  LMR_HISTORY_SCALE: {C.LMR_HISTORY_SCALE}')
print(f'  HISTORY_BONUS_SCALE: {C.HISTORY_BONUS_SCALE}')
print(f'  HISTORY_BONUS_CAP: {C.HISTORY_BONUS_CAP}')
print(f'  HISTORY_MALUS_CAP: {C.HISTORY_MALUS_CAP}')
print(f'  LMR_DIVISOR_MIN_DEPTH: {getattr(C, "LMR_DIVISOR_MIN_DEPTH", None)}')
print(f'  LMR_DIVISOR_DEPTH_STEP: {getattr(C, "LMR_DIVISOR_DEPTH_STEP", None)}')

assert C.TUNE_LMR_BASE_OFFSET == 6, f'CRITICAL: TUNE_LMR_BASE_OFFSET is {C.TUNE_LMR_BASE_OFFSET}, expected 6!'
assert C.LMR_BASE_OFFSET == 449, f'LMR_BASE_OFFSET mismatch: {C.LMR_BASE_OFFSET}'
assert C.HISTORY_MALUS_CAP == 1600, f'HISTORY_MALUS_CAP mismatch: {C.HISTORY_MALUS_CAP}'
print('\n🎉 所有常數與保護機制校驗 100% 正確！')


✅ 引擎目前生效參數檢驗：
  TUNE_LMR_BASE_OFFSET (slot index): 6
  LMR_BASE_OFFSET: 449
  LMR_MOVECOUNT_FACTOR: 29
  LMR_HISTORY_SCALE: 200
  HISTORY_BONUS_SCALE: 137
  HISTORY_BONUS_CAP: 2026
  HISTORY_MALUS_CAP: 1600
  LMR_DIVISOR_MIN_DEPTH: 10
  LMR_DIVISOR_DEPTH_STEP: 128

🎉 所有常數與保護機制校驗 100% 正確！


In [13]:
# 4. 執行對抗錦標賽（300k nodes, 400 局, concurrency 12, TT 64MB）
%cd /content/chess_numba
!python tools/tournament.py --games 400 --nodes 300000 --concurrency 7 --tt-mb 64


/content/chess_numba
Backend: inprocess; workers=7; TT=64 MB/context; openings=34700; seed=0
JIT plan: classical once + classical_old once; worker contexts share compiled code.
Warm-up: Kiwipete depth 14 for each Python/Numba backend, then one complete startpos game at depth 1 (all results discarded).
[pool] allocate worker 1/7
[warmup] New: Kiwipete depth=14...
info depth 1 score cp 1 nodes 494 nps 1166 hashfull 0.0% time 423 pv e2a6
info depth 2 score cp 1 nodes 885 nps 1339 hashfull 0.0% time 660 pv e2a6 b4c3
info depth 3 score cp -34 nodes 1859 nps 2808 hashfull 0.0% time 661 pv e2a6 b4c3 d2c3
info depth 4 score cp -34 nodes 2297 nps 3467 hashfull 0.0% time 662 pv e2a6 b4c3 d2c3 h3g2
info depth 5 score cp -125 nodes 5717 nps 8580 hashfull 0.0% time 666 pv e2a6 b4c3 d2c3 e6d5 e4d5
info depth 6 score cp -125 nodes 6293 nps 9433 hashfull 0.0% time 667 pv e2a6 b4c3 d2c3 e6d5 e4d5 h3g2
info depth 7 score cp -91 nodes 7940 nps 11861 hashfull 0.0% time 669 pv e2a6 b4c3 d2c3 e6d5 e5g4 f6g4

In [ ]:
# 5. 檢視對戰結果與下載 PGN 記錄
from pathlib import Path
import glob

pgn_files = glob.glob("tournament_analysis/*.pgn")
if pgn_files:
    latest_pgn = max(pgn_files, key=os.path.getmtime)
    print(f"最新 PGN 檔案: {latest_pgn} (大小: {os.path.getsize(latest_pgn)} bytes)")
    
    # 顯示前 3 場棋譜頭部資訊
    with open(latest_pgn, 'r', encoding='utf-8') as f:
        preview = ''.join([f.readline() for _ in range(35)])
        print("\n--- PGN 預覽 ---\n" + preview)
        
    # Colab 提供一鍵下載
    try:
        from google.colab import files
        files.download(latest_pgn)
    except ImportError:
        pass
else:
    print("尚未找到 PGN 記錄檔。")
